# DiffCFD — Differentiable Shape Optimization

This notebook demonstrates gradient-based optimization through the steady-state
SIMPLE solver using implicit differentiation (C1 patent claim).

We optimize the inlet velocity to minimize pressure drop in a Poiseuille channel,
demonstrating that `loss.backward()` gives exact analytical gradients through the
converged SIMPLE solution via matrix-free GMRES.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffcfd import NavierStokes2D

## Implicit Differentiation Demo

We set up a Poiseuille channel flow where `inlet_velocity` is a differentiable parameter.
The implicit diff backward uses matrix-free GMRES to compute exact gradients in O(N) memory.

In [ ]:
# Create solver with implicit differentiation
solver = NavierStokes2D(
    reynolds_number=1.0,
    grid=(32, 16),
    lx=4.0,
    ly=1.0,
    backward="implicit_diff",
    tol=1e-8,
)

# Differentiable inlet velocity
u_inlet = torch.tensor(1.0, requires_grad=True)

# Solve and compute loss
ux, uy, p = solver.solve_steady(sdf=None, inlet_velocity=u_inlet, case="channel")
pressure_drop = solver.pressure_drop(ux, uy, p)

# Backward pass: implicit differentiation through SIMPLE
pressure_drop.backward()

print(f"Pressure drop: {pressure_drop.item():.4f}")
print(f"∂ΔP/∂U_inlet (implicit diff): {u_inlet.grad.item():.4f}")
print("Analytical reference: 12.0 (for Re=1 Poiseuille)")

## Optimization Loop

Now we run a simple optimization: find the inlet velocity that achieves a target pressure drop.

In [ ]:
target_dp = 15.0  # Target pressure drop

u_opt = torch.tensor(0.5, requires_grad=True)
optimizer = torch.optim.Adam([u_opt], lr=0.1)

dp_history = []
u_history = []

for step in range(30):
    optimizer.zero_grad()
    ux, uy, p = solver.solve_steady(sdf=None, inlet_velocity=u_opt, case="channel")
    dp = solver.pressure_drop(ux, uy, p)
    loss = (dp - target_dp) ** 2
    loss.backward()
    optimizer.step()

    dp_history.append(dp.item())
    u_history.append(u_opt.item())

    if step % 5 == 0:
        print(
            f"Step {step}: U={u_opt.item():.4f}, ΔP={dp.item():.4f}, loss={loss.item():.2e}"
        )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(dp_history, "b-o", markersize=3)
ax1.axhline(y=target_dp, color="r", linestyle="--", label=f"Target ΔP={target_dp}")
ax1.set_xlabel("Optimization step")
ax1.set_ylabel("Pressure drop")
ax1.set_title("Pressure drop convergence")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(u_history, "g-o", markersize=3)
ax2.set_xlabel("Optimization step")
ax2.set_ylabel("Inlet velocity")
ax2.set_title("Design parameter evolution")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()